# Qubit-Count Scaling for ML-QRC

This notebook implements the **Qubit-Count Scaling** subsection from the draft.

It reuses the same Mackey-Glass forecasting setup as the main Task 1 notebook, but varies the number of reservoir qubits:

- `n ∈ {4, 6, 8, 10}`
- same Mackey-Glass series and train/test split
- same reservoir architecture, same reservoir seed, same noise level
- same proposed readout: **ML-QRC-M** with `d = 8`
- comparison baseline: **QRC + Ridge**

The main reported quantity is

\[
\Delta \mathrm{NMSE}(n) = \mathrm{NMSE}_{\mathrm{Ridge}}(n) - \mathrm{NMSE}_{\mathrm{ML\text{-}QRC\text{-}M}}(n),
\]

so a **positive** value means the proposed ML-QRC readout is better.


In [ ]:
# This cell installs the packages used by the notebook.
# The list is intentionally short and matches the rest of the project.

%pip install -q pennylane torch numpy pandas matplotlib scipy jupyter


In [ ]:
# This cell imports the libraries, resolves the repository root, and
# prepares the artifact directory used by the notebook.

from pathlib import Path
import gc
import os
import random
import time

here = Path.cwd()
if (here / "scripts").exists():
    ROOT = here
elif (here.parent / "scripts").exists():
    ROOT = here.parent
else:
    ROOT = here

cache_root = ROOT / "notebooks" / "artifacts" / "cache"
cache_root.mkdir(parents=True, exist_ok=True)
(cache_root / "mplconfig").mkdir(parents=True, exist_ok=True)

os.environ["XDG_CACHE_HOME"] = str(cache_root.resolve())
os.environ["MPLCONFIGDIR"] = str((cache_root / "mplconfig").resolve())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pennylane as qml
import torch

from scipy.stats import pearsonr
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

plt.rcParams["figure.dpi"] = 1000
plt.rcParams["savefig.dpi"] = 1000
plt.rcParams["figure.figsize"] = (3, 2.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

pd.set_option("display.precision", 6)
torch.set_num_threads(1)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch device: {DEVICE}")

ARTIFACT_DIR = ROOT / "notebooks" / "artifacts" / "qubit_scaling"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print(f"Repo root: {ROOT.resolve()}")
print(f"Artifacts: {ARTIFACT_DIR.resolve()}")
print(f"PyTorch: {torch.__version__}")
print(f"PennyLane: {qml.__version__}")


## Relation To The Existing Main Notebooks

This notebook is intentionally a **subsection notebook**.

- [`task1_mackey_glass_ml_qrc.ipynb`](./task1_mackey_glass_ml_qrc.ipynb) is the source notebook for the main Mackey-Glass Task 1 results.
- [`task2_tfim_ml_qrc.ipynb`](./task2_tfim_ml_qrc.ipynb) is the source notebook for the TFIM phase-classification results.
- This notebook extends the Task 1 setup with a qubit-count sweep so the draft can populate the qubit-scaling table and figure.

The `n = 6` row here should therefore line up with the full-data `Ridge` and `ML-QRC-M` values from Task 1.


In [ ]:
# This cell collects the experiment configuration in one place.
# The values mirror Task 1 whenever possible so the n=6 case stays
# directly comparable to the main Mackey-Glass notebook.

CONFIG = {
    # reproducibility
    "eval_seeds":       list(range(10)),
    "reservoir_seed":   42,
    # Mackey-Glass
    "num_points":       5000,
    "washout":          1000,
    "train_pool":       3000,
    "test_size":        1000,
    "mg_beta":          0.2,
    "mg_gamma":         0.1,
    "mg_q":             10,
    "mg_tau":           17.0,
    "mg_step":          0.1,
    "mg_x0":            1.2,
    # qubit scaling
    "n_values":         [4, 6, 8, 10],
    "n_layers":         1,
    "lookback":         50,
    "p_dep":            0.01,
    "qrc_batch":        256,
    # proposed ML-QRC-M
    "d":                8,
    # training
    "epochs":           300,
    "batch_size":       32,
    "lr":               1e-3,
    "weight_decay":     1e-5,
}

T = CONFIG["lookback"]
print(pd.Series(CONFIG, name="value"))


In [ ]:
# This cell generates the Mackey-Glass trajectory and the fixed
# train/test split used by every qubit-count setting.

def generate_mackey_glass(num_points, beta, gamma, q, tau, step, x0):
    delay = int(round(tau / step))
    total = delay + num_points
    x = np.full(total + 1, x0, dtype=np.float64)

    def rhs(x_now, x_delay):
        return beta * x_delay / (1.0 + x_delay ** q) - gamma * x_now

    for k in range(delay, total):
        x_now = x[k]
        x_d0 = x[k - delay]
        x_dh = 0.5 * (x[k - delay] + x[k - delay + 1])
        x_d1 = x[k - delay + 1]

        k1 = rhs(x_now, x_d0)
        k2 = rhs(x_now + 0.5 * step * k1, x_dh)
        k3 = rhs(x_now + 0.5 * step * k2, x_dh)
        k4 = rhs(x_now + step * k3, x_d1)

        x[k + 1] = x_now + (step / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)

    return x[delay + 1 : delay + 1 + num_points].astype(np.float32)

mg = generate_mackey_glass(
    CONFIG["num_points"],
    CONFIG["mg_beta"],
    CONFIG["mg_gamma"],
    CONFIG["mg_q"],
    CONFIG["mg_tau"],
    CONFIG["mg_step"],
    CONFIG["mg_x0"],
)

W = CONFIG["washout"]
TR = CONFIG["train_pool"]
TE = CONFIG["test_size"]

scale_min = float(mg[W:W + TR].min())
scale_range = float(mg[W:W + TR].max() - mg[W:W + TR].min())
mg_scaled = ((mg - scale_min) / scale_range).astype(np.float32)

def unscale(values):
    return (np.asarray(values, dtype=np.float64) * scale_range + scale_min).astype(np.float32)

decision_times = np.arange(T, len(mg), dtype=np.int32)
input_windows = np.stack([mg_scaled[i - T : i] for i in decision_times]).astype(np.float32)
targets_scaled = mg_scaled[decision_times].astype(np.float32)
targets_original = mg[decision_times].astype(np.float32)

train_mask = (decision_times >= W) & (decision_times < W + TR)
test_mask = (decision_times >= W + TR) & (decision_times < W + TR + TE)
train_positions = np.where(train_mask)[0]
test_positions = np.where(test_mask)[0]

assert len(train_positions) == TR
assert len(test_positions) == TE

print(f"Series length: {len(mg):,}")
print(f"Training windows: {len(train_positions):,}")
print(f"Test windows: {len(test_positions):,}")

fig, ax = plt.subplots(figsize=(3, 2.5), dpi=1000)
ax.plot(mg, color="black", linewidth=0.5)
ax.axvspan(0, W, color="#d9d9d9", alpha=0.6, label="Washout")
ax.axvspan(W, W + TR, color="#a1d99b", alpha=0.4, label="Train")
ax.axvspan(W + TR, W + TR + TE, color="#fdae6b", alpha=0.4, label="Test")
ax.set_xlabel("Time index")
ax.set_ylabel("x(t)")
ax.set_title("Mackey-Glass series")
ax.legend(fontsize=5, frameon=False, ncol=1)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "mg_series.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# This cell defines the quantum-reservoir feature builder.
# The logic matches Task 1, but it is written as a function of the
# qubit count so we can rebuild the reservoir at n = 4, 6, 8, 10.
#
# To keep the larger n cases tractable, the features are cached as a
# memory-mapped .npy file and reopened lazily when needed.

def observable_count(n_qubits: int) -> int:
    return n_qubits + n_qubits * (n_qubits - 1) // 2

def mlqrc_param_count(n_qubits: int, d: int, time_steps: int) -> int:
    f_dim = observable_count(n_qubits)
    return d * time_steps + d * n_qubits + d * f_dim + d ** 3 + 1

def build_qrc_feature_cache(
    windows: np.ndarray,
    n_qubits: int,
    time_steps: int,
    n_layers: int,
    reservoir_seed: int,
    p_dep: float,
    batch_size: int,
    artifact_dir: Path,
) -> Path:
    f_local = n_qubits
    f_pair = n_qubits * (n_qubits - 1) // 2
    f_dim = f_local + f_pair

    cache_path = artifact_dir / (
        f"qrc_feats_rseed{reservoir_seed}_p{p_dep}_q{n_qubits}_T{time_steps}.npy"
    )

    expected_shape = (len(windows), time_steps, n_qubits, f_dim)
    if cache_path.exists():
        cached = np.load(cache_path, mmap_mode="r")
        if cached.shape == expected_shape:
            print(f"Loading cached QRC features: {cache_path.name}")
            print(f"  shape: {cached.shape}")
            del cached
            return cache_path

    print(f"Computing QRC features for n={n_qubits} ...")

    rng_res = np.random.default_rng(reservoir_seed)
    reservoir_weights = rng_res.uniform(
        0.0,
        2.0 * np.pi,
        size=(n_layers, n_qubits, 3),
    ).astype(np.float32)

    reservoir_unitary = torch.tensor(
        qml.matrix(
            qml.StronglyEntanglingLayers(reservoir_weights, wires=range(n_qubits)),
            wire_order=range(n_qubits),
        ),
        dtype=torch.complex64,
        device=DEVICE,
    )

    def observable_matrix(op):
        return torch.tensor(qml.matrix(op, wire_order=range(n_qubits)), dtype=torch.complex64, device=DEVICE)

    observable_matrices = [observable_matrix(qml.PauliZ(i)) for i in range(n_qubits)]
    for i in range(n_qubits):
        for j in range(i + 1, n_qubits):
            observable_matrices.append(observable_matrix(qml.PauliZ(i) @ qml.PauliZ(j)))

    noise_local = float(1.0 - 4.0 * p_dep / 3.0)
    noise_pair = float((1.0 - 4.0 * p_dep / 3.0) ** 2)

    zero_state = torch.zeros(2 ** n_qubits, dtype=torch.complex64, device=DEVICE)
    zero_state[0] = 1.0
    pi_tensor = torch.tensor(float(np.pi), dtype=torch.float32, device=DEVICE)
    qubit_indices = torch.arange(n_qubits, device=DEVICE)

    def batched_ry(states, angles, wire):
        c = torch.cos(angles / 2).to(torch.complex64)
        s = torch.sin(angles / 2).to(torch.complex64)
        gate = torch.stack(
            [
                torch.stack([c, -s], dim=-1),
                torch.stack([s, c], dim=-1),
            ],
            dim=-2,
        )
        shaped = states.reshape(states.shape[0], *([2] * n_qubits))
        shaped = torch.movedim(shaped, 1 + wire, -1)
        shaped = torch.einsum("bij,b...j->b...i", gate, shaped)
        shaped = torch.movedim(shaped, -1, 1 + wire)
        return shaped.reshape(states.shape[0], -1)

    output = np.lib.format.open_memmap(
        cache_path,
        mode="w+",
        dtype=np.float32,
        shape=expected_shape,
    )

    window_tensor = torch.tensor(windows, dtype=torch.float32, device=DEVICE)
    start_time = time.time()

    for start in range(0, len(window_tensor), batch_size):
        batch = window_tensor[start : start + batch_size]
        batch_size_now = batch.shape[0]
        states = zero_state.unsqueeze(0).expand(batch_size_now, -1).clone()
        features = torch.zeros(batch_size_now, time_steps, n_qubits, f_dim, dtype=torch.float32, device=DEVICE)

        for time_index in range(time_steps):
            angles = pi_tensor * batch[:, time_index]
            for wire in range(n_qubits):
                states = batched_ry(states, angles, wire)

            states = states @ reservoir_unitary.T
            expectations = torch.stack(
                [
                    torch.real(torch.sum(states.conj() * (states @ obs.T), dim=1))
                    for obs in observable_matrices
                ],
                dim=1,
            )

            expectations[:, :f_local] *= noise_local
            expectations[:, f_local:] *= noise_pair

            features[:, time_index, qubit_indices, qubit_indices] = expectations[:, :f_local]
            pair_index = 0
            for i in range(n_qubits):
                for j in range(i + 1, n_qubits):
                    column = f_local + pair_index
                    features[:, time_index, i, column] = expectations[:, column]
                    features[:, time_index, j, column] = expectations[:, column]
                    pair_index += 1

        output[start : start + batch_size_now] = features.cpu().numpy()

        done = min(start + batch_size_now, len(window_tensor))
        if done % 1024 == 0 or done == len(window_tensor):
            elapsed = time.time() - start_time
            print(f"  {done}/{len(window_tensor)} windows  ({elapsed:.0f}s)")

    output.flush()
    del output
    print(f"Saved QRC features to: {cache_path.resolve()}")
    return cache_path


In [ ]:
# This cell defines the common helpers and the proposed ML-QRC-M model.
# The training loop is kept simple and follows the same setup as Task 1.

def nmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2))

class Standardizer:
    def fit(self, features):
        flat = features.reshape(features.shape[0], -1)
        self.mean = flat.mean(axis=0, dtype=np.float64)
        self.std = flat.std(axis=0, dtype=np.float64)
        self.std[self.std < 1e-8] = 1.0
        return self

    def transform(self, features):
        shape = features.shape
        flat = features.reshape(shape[0], -1)
        flat = (flat - self.mean) / self.std
        return flat.reshape(shape).astype(np.float32)

def fit_ols(features, targets):
    design = np.hstack([features, np.ones((len(features), 1), dtype=np.float32)])
    return np.linalg.lstsq(design, targets, rcond=None)[0].astype(np.float32)

def predict_ols(weights, features):
    design = np.hstack([features, np.ones((len(features), 1), dtype=np.float32)])
    return (design @ weights).astype(np.float32)

RIDGE_ALPHAS = (1e-4, 1e-3, 1e-2, 0.1, 1.0, 10.0, 100.0, 1e3, 1e4, 1e5)

def fit_ridge_cv(features, targets, alphas=RIDGE_ALPHAS, val_frac=0.2):
    split = max(1, int(len(features) * (1.0 - val_frac)))
    x_train = features[:split].astype(np.float64)
    y_train = targets[:split].astype(np.float64)
    x_val = features[split:].astype(np.float64)
    y_val = targets[split:].astype(np.float64)

    mean_x = x_train.mean(axis=0)
    mean_y = y_train.mean()
    centered_x = x_train - mean_x
    centered_y = y_train - mean_y

    best_alpha = None
    best_score = np.inf

    for alpha in alphas:
        dual = np.linalg.solve(
            centered_x @ centered_x.T + alpha * np.eye(len(x_train)),
            centered_y,
        )
        weights = centered_x.T @ dual
        bias = mean_y - mean_x @ weights
        score = nmse(y_val, x_val @ weights + bias)
        if score < best_score:
            best_score = score
            best_alpha = alpha

    full_mean_x = features.astype(np.float64).mean(axis=0)
    full_mean_y = float(targets.mean())
    full_centered_x = features.astype(np.float64) - full_mean_x
    full_centered_y = targets.astype(np.float64) - full_mean_y

    dual = np.linalg.solve(
        full_centered_x @ full_centered_x.T + best_alpha * np.eye(len(features)),
        full_centered_y,
    )
    weights = full_centered_x.T @ dual
    bias = full_mean_y - full_mean_x @ weights
    return np.append(weights, bias).astype(np.float32), best_alpha

class MLQRCReadout(nn.Module):
    def __init__(self, time_steps, n_qubits, f_dim, d):
        super().__init__()
        self.u_t = nn.Parameter(torch.randn(d, time_steps) * 0.05)
        self.u_n = nn.Parameter(torch.randn(d, n_qubits) * 0.05)
        self.u_f = nn.Parameter(torch.randn(d, f_dim) * 0.05)
        self.head = nn.Linear(d * d * d, 1)

    def forward(self, x):
        y = torch.einsum("btnf,dt->bdnf", x, self.u_t)
        y = torch.einsum("bdnf,in->bdif", y, self.u_n)
        y = torch.einsum("bdif,jf->bdij", y, self.u_f)
        return self.head(y.reshape(y.shape[0], -1)).squeeze(-1)

def count_parameters(model):
    return int(sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad))

def train_model(model, features, targets, epochs, batch_size, lr, weight_decay):
    model = model.to(DEVICE)
    dataset = TensorDataset(
        torch.from_numpy(features),
        torch.from_numpy(targets.astype(np.float32)),
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )
    loss_function = nn.MSELoss()
    model.train()

    for _ in range(epochs):
        for batch_features, batch_targets in loader:
            batch_features = batch_features.to(DEVICE)
            batch_targets = batch_targets.to(DEVICE)
            optimizer.zero_grad()
            predictions = model(batch_features)
            loss = loss_function(predictions, batch_targets)
            loss.backward()
            optimizer.step()

    return model

@torch.no_grad()
def predict_model(model, features):
    model.eval()
    tensor_features = torch.from_numpy(features).to(DEVICE)
    return model(tensor_features).cpu().numpy().astype(np.float32)

print("Helpers ready.")


In [ ]:
# This cell runs the full qubit-count scaling experiment.
# For each n value we:
#   1. rebuild or load the QRC tensor bank,
#   2. fit Ridge once on the full-data regime,
#   3. train ML-QRC-M for 10 seeds,
#   4. store the per-seed delta NMSE values.

raw_rows = []
summary_rows = []

for n_qubits in CONFIG["n_values"]:
    print("=" * 88)
    print(f"Qubit count n = {n_qubits}")

    cache_path = build_qrc_feature_cache(
        windows=input_windows,
        n_qubits=n_qubits,
        time_steps=T,
        n_layers=CONFIG["n_layers"],
        reservoir_seed=CONFIG["reservoir_seed"],
        p_dep=CONFIG["p_dep"],
        batch_size=CONFIG["qrc_batch"],
        artifact_dir=ARTIFACT_DIR,
    )

    feature_bank = np.load(cache_path, mmap_mode="r")
    f_dim = feature_bank.shape[3]
    flat_dim = T * n_qubits * f_dim

    train_raw = np.asarray(feature_bank[train_positions], dtype=np.float32)
    test_raw = np.asarray(feature_bank[test_positions], dtype=np.float32)
    del feature_bank
    gc.collect()

    scaler = Standardizer().fit(train_raw)
    train_tensor = scaler.transform(train_raw)
    test_tensor = scaler.transform(test_raw)
    del train_raw, test_raw
    gc.collect()

    train_flat = train_tensor.reshape(len(train_positions), -1)
    test_flat = test_tensor.reshape(len(test_positions), -1)
    train_targets = targets_scaled[train_positions]
    test_targets = targets_original[test_positions]

    ridge_weights, ridge_alpha = fit_ridge_cv(train_flat, train_targets)
    ridge_predictions = unscale(predict_ols(ridge_weights, test_flat))
    ridge_score = nmse(test_targets, ridge_predictions)

    print(f"  F = {f_dim}")
    print(f"  Flat dimension = {flat_dim:,}")
    print(f"  Ridge alpha = {ridge_alpha}")
    print(f"  Ridge NMSE = {ridge_score:.6f}")
    print(f"  ML-QRC-M parameters = {mlqrc_param_count(n_qubits, CONFIG['d'], T):,}")

    mlqrc_scores = []
    for seed in CONFIG["eval_seeds"]:
        print(f"    Seed {seed}: training ML-QRC-M")
        set_seed(seed)
        model = MLQRCReadout(T, n_qubits, f_dim, CONFIG["d"])
        model = train_model(
            model,
            train_tensor,
            train_targets,
            CONFIG["epochs"],
            CONFIG["batch_size"],
            CONFIG["lr"],
            CONFIG["weight_decay"],
        )
        predictions = unscale(predict_model(model, test_tensor))
        score = nmse(test_targets, predictions)
        mlqrc_scores.append(score)
        raw_rows.append(
            {
                "n_qubits": n_qubits,
                "F": f_dim,
                "flat_dim": flat_dim,
                "seed": seed,
                "ridge_nmse": ridge_score,
                "mlqrc_nmse": score,
                "delta_nmse": ridge_score - score,
            }
        )
        del model, predictions
        gc.collect()

    mlqrc_scores = np.asarray(mlqrc_scores, dtype=np.float64)
    deltas = ridge_score - mlqrc_scores
    summary_rows.append(
        {
            "n_qubits": n_qubits,
            "F": f_dim,
            "flat_dim": flat_dim,
            "ridge_mean": ridge_score,
            "ridge_std": 0.0,
            "mlqrc_mean": float(mlqrc_scores.mean()),
            "mlqrc_std": float(mlqrc_scores.std()),
            "delta_mean": float(deltas.mean()),
            "delta_std": float(deltas.std()),
        }
    )

    del train_tensor, test_tensor, train_flat, test_flat
    gc.collect()

raw_results = pd.DataFrame(raw_rows)
summary_results = pd.DataFrame(summary_rows)

raw_path = ARTIFACT_DIR / "qubit_scaling_raw.csv"
table_path = ARTIFACT_DIR / "table_qubit_scaling.csv"
raw_results.to_csv(raw_path, index=False, float_format="%.6f")
summary_results.to_csv(table_path, index=False, float_format="%.6f")

print()
print("Raw per-seed results:")
display(raw_results.head(12))
print()
print("Summary results:")
display(summary_results)
print(f"Saved raw results to: {raw_path.resolve()}")
print(f"Saved summary table to: {table_path.resolve()}")


In [ ]:
# This cell formats the summary table in a paper-friendly way and
# prints LaTeX rows for quick copy into the draft.

table_view = summary_results.copy()
table_view["Ridge"] = table_view.apply(
    lambda row: f"{row['ridge_mean']:.4f} ± {row['ridge_std']:.4f}",
    axis=1,
)
table_view["ML-QRC-M"] = table_view.apply(
    lambda row: f"{row['mlqrc_mean']:.4f} ± {row['mlqrc_std']:.4f}",
    axis=1,
)
table_view["ΔNMSE"] = table_view.apply(
    lambda row: f"{row['delta_mean']:.4f} ± {row['delta_std']:.4f}",
    axis=1,
)
table_view = table_view[["n_qubits", "F", "flat_dim", "Ridge", "ML-QRC-M", "ΔNMSE"]]
table_view = table_view.rename(
    columns={"n_qubits": "n", "flat_dim": "Flat dim"}
)

print("=" * 92)
print(" TABLE V  Qubit-count scaling on Mackey-Glass  (NMSE, 10 seeds)")
print("=" * 92)
print(f"  {'n':>2}  {'F':>3}  {'Flat dim':>10}  {'Ridge':>18}  {'ML-QRC-M':>18}  {'ΔNMSE':>18}")
print("-" * 92)
for _, row in summary_results.iterrows():
    print(
        f"  {int(row['n_qubits']):>2}  "
        f"{int(row['F']):>3}  "
        f"{int(row['flat_dim']):>10,}  "
        f"{row['ridge_mean']:.4f} ± {row['ridge_std']:.4f}  "
        f"{row['mlqrc_mean']:.4f} ± {row['mlqrc_std']:.4f}  "
        f"{row['delta_mean']:.4f} ± {row['delta_std']:.4f}"
    )
print("=" * 92)

print("\nLaTeX rows:")
for _, row in summary_results.iterrows():
    print(
        f"    {int(row['n_qubits'])} & {int(row['F'])} & {int(row['flat_dim']):,} & "
        f"${row['ridge_mean']:.4f}\\pm{row['ridge_std']:.4f}$ & "
        f"${row['mlqrc_mean']:.4f}\\pm{row['mlqrc_std']:.4f}$ & "
        f"${row['delta_mean']:.4f}\\pm{row['delta_std']:.4f}$ \\"
    )

display(table_view)


In [ ]:
# This cell makes the main draft figure: ΔNMSE versus qubit count.
# Positive values mean ML-QRC-M is better than Ridge.

pearson_r, pearson_p = pearsonr(summary_results["n_qubits"], summary_results["delta_mean"])
print(f"Pearson r = {pearson_r:.4f}")
print(f"Pearson p = {pearson_p:.4f}")

fig, ax = plt.subplots(figsize=(3, 2.5), dpi=1000)
ax.errorbar(
    summary_results["n_qubits"],
    summary_results["delta_mean"],
    yerr=summary_results["delta_std"],
    fmt="o-",
    color="#08519c",
    linewidth=1.4,
    markersize=4.5,
    capsize=3,
)
ax.axhline(0.0, color="black", linestyle="--", linewidth=0.8)
ax.set_xlabel("Qubit count n")
ax.set_ylabel("ΔNMSE = Ridge - ML-QRC")
ax.set_title("Qubit-count scaling")
ax.set_xticks(summary_results["n_qubits"])
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "fig_qubit_scaling.pdf", bbox_inches="tight")
plt.savefig(ARTIFACT_DIR / "fig_qubit_scaling.png", bbox_inches="tight")
plt.show()
print("Saved: fig_qubit_scaling.pdf")


In [ ]:
# This cell gives a direct model-vs-model NMSE view, which is useful
# for checking where the delta values come from.

fig, ax = plt.subplots(figsize=(3, 2.5), dpi=1000)
ax.errorbar(
    summary_results["n_qubits"],
    summary_results["ridge_mean"],
    yerr=summary_results["ridge_std"],
    fmt="s--",
    color="#74c476",
    linewidth=1.1,
    markersize=4,
    capsize=3,
    label="Ridge",
)
ax.errorbar(
    summary_results["n_qubits"],
    summary_results["mlqrc_mean"],
    yerr=summary_results["mlqrc_std"],
    fmt="o-",
    color="#005a32",
    linewidth=1.4,
    markersize=4.5,
    capsize=3,
    label="ML-QRC-M",
)
ax.set_xlabel("Qubit count n")
ax.set_ylabel("NMSE")
ax.set_title("Ridge vs ML-QRC-M")
ax.set_xticks(summary_results["n_qubits"])
ax.legend(fontsize=6, frameon=False)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "fig_qubit_scaling_models.pdf", bbox_inches="tight")
plt.savefig(ARTIFACT_DIR / "fig_qubit_scaling_models.png", bbox_inches="tight")
plt.show()
print("Saved: fig_qubit_scaling_models.pdf")


In [ ]:
# This cell checks the n=6 row against the already executed Task 1
# notebook, if the Task 1 artifact table is available locally.

reference_path = ROOT / "notebooks" / "artifacts" / "task1_mg_v2" / "table2_main.csv"
if reference_path.exists():
    reference_table = pd.read_csv(reference_path)
    ridge_ref = float(reference_table.loc[reference_table["Method"] == "Ridge", "full_mean"].iloc[0])
    mlqrc_ref = float(reference_table.loc[reference_table["Method"] == "ML-QRC (ours)", "full_mean"].iloc[0])

    current_row = summary_results.loc[summary_results["n_qubits"] == 6].iloc[0]
    comparison = pd.DataFrame(
        [
            {"Source": "Task 1 notebook", "Ridge": ridge_ref, "ML-QRC-M": mlqrc_ref},
            {
                "Source": "Qubit-scaling notebook",
                "Ridge": float(current_row["ridge_mean"]),
                "ML-QRC-M": float(current_row["mlqrc_mean"]),
            },
        ]
    )
    display(comparison)
else:
    print("Task 1 reference artifact not found. Skipping n=6 consistency check.")


## Summary

This notebook is the draft-ready implementation of the **Qubit-Count Scaling** subsection.

It should be used together with:

- [`task1_mackey_glass_ml_qrc.ipynb`](./task1_mackey_glass_ml_qrc.ipynb) for the main Mackey-Glass task,
- [`task2_tfim_ml_qrc.ipynb`](./task2_tfim_ml_qrc.ipynb) for the TFIM phase-classification task.

The key outputs saved by this notebook are:

- `table_qubit_scaling.csv`
- `qubit_scaling_raw.csv`
- `fig_qubit_scaling.pdf`
- `fig_qubit_scaling_models.pdf`
